# Stage 22: Interaction/Cancellation Decomposition

Stage 21 showed a striking result: $E_m \gg E_{\text{full}}$. 
Applying the coordinate intervention to ONLY the first moment causes an enormous deviation in the update vector. But applying it to BOTH the first and second moments simultaneously results in a much smaller deviation. 

This implies that the coupled second-moment normalization interacts highly non-linearly to cancel out the first-moment explosion.

We mathematically define this non-additive interaction as:
$\Delta_m = d_m - d_{\text{nat}}$
$\Delta_v = d_v - d_{\text{nat}}$
$\Delta_{\text{full}} = d_{\text{full}} - d_{\text{nat}}$
$\Delta_{\text{int}} = \Delta_{\text{full}} - \Delta_m - \Delta_v$

This notebook records the relative magnitudes of these components against $\|d_{\text{nat}}\|$, and measures their angular interference $\cos(\Delta_m, \Delta_v)$, $\cos(\Delta_m, \Delta_{\text{int}})$, $\cos(\Delta_v, \Delta_{\text{int}})$. If the attenuation is an active cancellation effect, we will see large interaction magnitude and strong negative angular correlations.

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

sys.path.append('.')
from official_soap import SOAP


In [ ]:
class SOAPStage22(SOAP):
    def __init__(self, params, lr=1e-3, betas=(0.95, 0.95), shampoo_beta=-1, 
                 eps=1e-8, weight_decay=0.0, precondition_frequency=10, 
                 max_precond_dim=10000, merge_dims=False, precondition_1d=False,
                 normalize_grads=False, data_format='channels_first', correct_bias=True,
                 mu=0.5, damping=1e-3):
        super().__init__(params, lr=lr, betas=betas, shampoo_beta=shampoo_beta, 
                         eps=eps, weight_decay=weight_decay, 
                         precondition_frequency=precondition_frequency,
                         max_precond_dim=max_precond_dim, merge_dims=merge_dims, 
                         precondition_1d=precondition_1d, normalize_grads=normalize_grads, 
                         data_format=data_format, correct_bias=correct_bias)
        self.mu = mu
        self.damping = damping
        self.layer_diagnostics = []
        self.target_steps = {50, 150, 300, 450}
        self.target_layers = {0, 2, 4, 6, 8}
        self.n_perm = 50

    def functional_soap_update_decomposed(self, g_nat, g_int, state, group, p):
        b1, b2 = group['betas']
        eps = group['eps']
        
        proj_nat = self.project(g_nat.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        proj_int = self.project(g_int.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        m_old = state.get("exp_avg", torch.zeros_like(proj_nat))
        v_old = state.get("exp_avg_sq", torch.zeros_like(proj_nat))
        
        m_nat = m_old * b1 + proj_nat * (1.0 - b1)
        v_nat = v_old * b2 + proj_nat.square() * (1.0 - b2)
        
        m_int = m_old * b1 + proj_int * (1.0 - b1)
        v_int = v_old * b2 + proj_int.square() * (1.0 - b2)
        
        denom_nat = v_nat.sqrt() + eps
        denom_int = v_int.sqrt() + eps
        
        step_val = state.get('step', 0) + 1
        step_size = group['lr']
        if getattr(self, "correct_bias", True):
            bias_correction1 = 1.0 - b1 ** step_val
            bias_correction2 = 1.0 - b2 ** step_val
            step_size = step_size * (math.sqrt(bias_correction2)) / bias_correction1
            
        def compute_d(m_new, denom):
            norm_grad = self.project_back(m_new / denom, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
            if group.get("normalize_grads", False):
                norm_grad = norm_grad / (1e-30 + torch.mean(norm_grad**2)**0.5)
            update = -step_size * norm_grad
            if group.get("weight_decay", 0.0) > 0.0:
                update = update - group['lr'] * group['weight_decay'] * p.data
            return update
            
        d_nat = compute_d(m_nat, denom_nat)
        d_m = compute_d(m_int, denom_nat)
        d_v = compute_d(m_nat, denom_int)
        d_full = compute_d(m_int, denom_int)
        
        return d_nat, d_m, d_v, d_full

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.layer_diagnostics = []
        
        current_step = 0
        if len(self.param_groups) > 0 and len(self.param_groups[0]['params']) > 0:
            p_sample = self.param_groups[0]['params'][0]
            if 'step' in self.state[p_sample]:
                current_step = self.state[p_sample]['step'] + 1
        else:
            current_step = 1
            
        idx = 0
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    idx += 1
                    continue
                    
                layer_idx = idx
                state = self.state[p]
                raw_g = p.grad.clone()
                
                if 'moq_initialized' not in state:
                    state['moq_initialized'] = True
                    state['prev_raw_g'] = raw_g.clone()
                    g_moq = raw_g.clone()
                else:
                    if current_step in self.target_steps and layer_idx in self.target_layers and p.dim() > 1 and 'Q' in state and state['Q'] is not None:
                        g_nat = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                        
                        proj_raw = self.project(raw_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                        b1, b2 = group["betas"]
                        v_old = state.get("exp_avg_sq")
                        
                        if v_old is not None:
                            v_raw = v_old * b2 + proj_raw.square() * (1.0 - b2)
                            denom_raw = v_raw.sqrt() + group["eps"]
                            D_inv = 1.0 / denom_raw
                            
                            delta_g_orig = self.mu * (raw_g - state['prev_raw_g'])
                            proj_delta = self.project(delta_g_orig.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                            
                            delta_m = proj_delta * (1.0 - b1)
                            delta_m_norm = float(torch.linalg.vector_norm(delta_m))
                            
                            if delta_m_norm > 1e-12:
                                denom_flat = denom_raw.flatten()
                                k10 = max(1, int(0.10 * denom_flat.numel()))
                                _, indices10 = torch.topk(denom_flat, k10, largest=False)
                                
                                tail_mask = torch.zeros_like(denom_flat, dtype=torch.bool)
                                tail_mask[indices10] = True
                                
                                z_delta = D_inv * delta_m
                                z_delta_sq = z_delta.flatten()**2
                                sum_z_delta_sq = float(torch.sum(z_delta_sq)) + 1e-12
                                F10_nat = float(torch.sum(z_delta_sq[indices10])) / sum_z_delta_sq
                                
                                delta_m_flat = delta_m.flatten()
                                d_T_vals = delta_m_flat[tail_mask]
                                d_N_vals = delta_m_flat[~tail_mask]
                                
                                if len(d_T_vals) > 0 and len(d_N_vals) > 0:
                                    E_m_list = []
                                    E_v_list = []
                                    E_full_list = []
                                    I_list = []
                                    
                                    cos_dm_dv_list = []
                                    cos_dm_dint_list = []
                                    cos_dv_dint_list = []
                                    
                                    def cos_sim(v1, v2):
                                        nv1 = float(torch.linalg.vector_norm(v1))
                                        nv2 = float(torch.linalg.vector_norm(v2))
                                        if nv1 < 1e-12 or nv2 < 1e-12:
                                            return 0.0
                                        return float(torch.sum(v1 * v2)) / (nv1 * nv2)
                                        
                                    for _ in range(self.n_perm):
                                        perm_T = torch.randperm(len(d_T_vals), device=d_T_vals.device)
                                        perm_N = torch.randperm(len(d_N_vals), device=d_N_vals.device)
                                        
                                        d_T_perm = d_T_vals[perm_T]
                                        d_N_perm = d_N_vals[perm_N]
                                        
                                        d_perm_flat = torch.zeros_like(delta_m_flat)
                                        d_perm_flat[tail_mask] = d_T_perm
                                        d_perm_flat[~tail_mask] = d_N_perm
                                        d_perm = d_perm_flat.view(delta_m.shape)
                                        
                                        tail_mask_shaped = tail_mask.view(denom_raw.shape)
                                        d_T_full = torch.where(tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                        d_N_full = torch.where(~tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                        
                                        norm_d_T_sq = float(torch.sum(d_T_full**2))
                                        norm_d_N_sq = float(torch.sum(d_N_full**2))
                                        
                                        E_T = float(torch.sum((D_inv * d_T_full)**2))
                                        E_N = float(torch.sum((D_inv * d_N_full)**2))
                                        
                                        if F10_nat == 0.0:
                                            c_T = 0.0
                                            c_N = 1.0 / math.sqrt(norm_d_N_sq + 1e-12)
                                        elif F10_nat == 1.0:
                                            c_T = 1.0 / math.sqrt(norm_d_T_sq + 1e-12)
                                            c_N = 0.0
                                        else:
                                            ratio = (F10_nat * E_N) / ((1.0 - F10_nat) * E_T + 1e-12)
                                            c_N_sq = 1.0 / (ratio * norm_d_T_sq + norm_d_N_sq + 1e-12)
                                            c_T_sq = c_N_sq * ratio
                                            c_N = math.sqrt(c_N_sq)
                                            c_T = math.sqrt(c_T_sq)
                                            
                                        d_corrected = (c_T * d_T_full + c_N * d_N_full) * delta_m_norm
                                        
                                        proj_delta_corrected = d_corrected / (1.0 - b1)
                                        delta_g_corrected = self.project_back(proj_delta_corrected, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                                        g_int = raw_g + delta_g_corrected
                                        
                                        d_nat, d_m, d_v, d_full = self.functional_soap_update_decomposed(g_nat, g_int, state, group, p)
                                        
                                        norm_d_nat = float(torch.linalg.vector_norm(d_nat))
                                        
                                        delta_m_up = d_m - d_nat
                                        delta_v_up = d_v - d_nat
                                        delta_full_up = d_full - d_nat
                                        delta_int_up = delta_full_up - delta_m_up - delta_v_up
                                        
                                        E_m_list.append(float(torch.linalg.vector_norm(delta_m_up)) / (norm_d_nat + 1e-12))
                                        E_v_list.append(float(torch.linalg.vector_norm(delta_v_up)) / (norm_d_nat + 1e-12))
                                        E_full_list.append(float(torch.linalg.vector_norm(delta_full_up)) / (norm_d_nat + 1e-12))
                                        I_list.append(float(torch.linalg.vector_norm(delta_int_up)) / (norm_d_nat + 1e-12))
                                        
                                        cos_dm_dv_list.append(cos_sim(delta_m_up, delta_v_up))
                                        cos_dm_dint_list.append(cos_sim(delta_m_up, delta_int_up))
                                        cos_dv_dint_list.append(cos_sim(delta_v_up, delta_int_up))
                                        
                                    diag = {
                                        "step": current_step,
                                        "layer": layer_idx,
                                        "norm_d_nat": norm_d_nat,
                                        "E_m_q25": float(np.percentile(E_m_list, 25)),
                                        "E_m_median": float(np.median(E_m_list)),
                                        "E_m_q75": float(np.percentile(E_m_list, 75)),
                                        "E_v_q25": float(np.percentile(E_v_list, 25)),
                                        "E_v_median": float(np.median(E_v_list)),
                                        "E_v_q75": float(np.percentile(E_v_list, 75)),
                                        "E_full_q25": float(np.percentile(E_full_list, 25)),
                                        "E_full_median": float(np.median(E_full_list)),
                                        "E_full_q75": float(np.percentile(E_full_list, 75)),
                                        "I_q25": float(np.percentile(I_list, 25)),
                                        "I_median": float(np.median(I_list)),
                                        "I_q75": float(np.percentile(I_list, 75)),
                                        "cos_dm_dv_median": float(np.median(cos_dm_dv_list)),
                                        "cos_dm_dint_median": float(np.median(cos_dm_dint_list)),
                                        "cos_dv_dint_median": float(np.median(cos_dv_dint_list))
                                    }
                                    self.layer_diagnostics.append(diag)
                    
                    # Always apply the natural update
                    g_moq = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    state['prev_raw_g'].copy_(raw_g)
                
                p.grad.copy_(g_moq)
                idx += 1
                
        super().step()
        return loss


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None
    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule

def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_stage22_audit(K: int=1, steps: int=500, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule_long.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
        if len(schedule) < steps:
            schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
            torch.save(schedule, schedule_path)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    opt = SOAPStage22(
        model.parameters(), 
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        mu=0.5, 
        damping=1e-3
    )

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        if len(opt.layer_diagnostics) > 0:
            history.extend(opt.layer_diagnostics)
            
        if step % 50 == 0 or step == 1:
            err = relative_l2(model, device)
            print(f"Step {step:3d} | Loss {loss.item():.4e} | RelL2 {err:.4e}")
            if not math.isfinite(loss.item()) or not math.isfinite(err):
                print(f"Divergence detected at step {step}")
                break

    return history


### Stage 22: Execute Interaction Decomposition

In [ ]:
import matplotlib.pyplot as plt

all_history = {}
for K in [10, 20]:
    print(f"\n{'='*40}\nRunning K={K}...\n{'='*40}")
    history = run_stage22_audit(K=K, steps=500)
    all_history[K] = history
    with open(f"stage22_K{K}_soap_moq.json", "w") as f:
        json.dump(history, f, indent=2)

# Main Key Plot
for K, history in all_history.items():
    E_m = [h["E_m_median"] for h in history]
    E_full = [h["E_full_median"] for h in history]
    I_int = [h["I_median"] for h in history]
    
    plt.figure(figsize=(8, 6))
    plt.scatter(E_m, E_full, alpha=0.7)
    plt.xlabel('E_m (First-Moment Perturbation)')
    plt.ylabel('E_full (Full SOAP Update)')
    plt.title(f'K={K}: E_m vs E_full')
    plt.grid(True)
    plt.savefig(f'stage22_K{K}_Em_vs_Efull.png')
    plt.close()
    
    plt.figure(figsize=(8, 6))
    plt.scatter(E_m, I_int, alpha=0.7)
    plt.xlabel('E_m (First-Moment Perturbation)')
    plt.ylabel('I (Interaction Magnitude)')
    plt.title(f'K={K}: E_m vs I')
    plt.grid(True)
    plt.savefig(f'stage22_K{K}_Em_vs_I.png')
    plt.close()
